# 01 — Exploratory Data Analysis

Sanity-check the dataset against the paper's reported counts:
- 27,895 traders tracked over 46 weeks
- 1,229,439 observations at the trader-week level

Also: keep_country investigation, missingness audit, column diagnostics.

In [ ]:
import os, sys
from pathlib import Path

def find_repo_root(start=None):
    """Walk up from the current directory to the folder with run_pipeline.py and src/."""
    here = Path(start or Path.cwd()).resolve()
    for d in [here, *here.parents]:
        if (d / 'run_pipeline.py').exists() and (d / 'src').is_dir():
            return d
    raise FileNotFoundError('Run this notebook from inside the repository.')

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

from social_trading_dml.data.loading import (
    load_raw_data, clean_data, verify_counts, print_cleaning_summary,
    save_processed, load_processed, verify_column_diagnostics,
    get_missingness_report, EXPECTED_OBS_PAPER, EXPECTED_TRADERS_PAPER,
)
from social_trading_dml.viz.figures import set_journal_style, COLORS

set_journal_style()
print('Imports OK')

## 1. Load Raw Data

In [ ]:
from social_trading_dml.config import default_raw_data_path
data_path = str(default_raw_data_path())
df_raw = load_raw_data(data_path)
print(f'Raw shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
print(f'Dtypes:\n{df_raw.dtypes}')

## 2. Missingness Report

In [ ]:
miss = get_missingness_report(df_raw)
print('=== COLUMNS WITH ANY MISSING VALUES ===')
print(miss[miss['n_missing'] > 0].to_string(index=False))
print()
print(f"learner_country_count: {miss.loc[miss['column']=='learner_country_count', 'pct_missing'].values[0]:.1%} missing → EXCLUDED")

## 3. Verify: Are joining_date/joining_date2/user_country_name nulls the same rows?

In [ ]:
jd_null = df_raw['joining_date'].isna()
jd2_null = df_raw['joining_date2'].isna() if 'joining_date2' in df_raw.columns else jd_null
ucn_null = df_raw['user_country_name'].isna()

print(f'joining_date null:       {jd_null.sum():>10,}')
print(f'joining_date2 null:      {jd2_null.sum():>10,}')
print(f'user_country_name null:  {ucn_null.sum():>10,}')
print(f'All three null (same):   {(jd_null & jd2_null & ucn_null).sum():>10,}')
print(f'Any of three null:       {(jd_null | jd2_null | ucn_null).sum():>10,}')

## 4. Check for Fully-Blank Trailer Rows

In [ ]:
fully_blank = df_raw.isnull().all(axis=1)
print(f'Fully-blank rows: {fully_blank.sum():,}')
if fully_blank.sum() > 0:
    print('Sample of fully-blank rows:')
    display(df_raw[fully_blank].head())
else:
    print('No fully-blank trailer rows found in this file.')

## 5. Excel Truncation Check

In [ ]:
n_rows = len(df_raw) - fully_blank.sum()
print(f'Data rows (non-blank): {n_rows:,}')
print(f'Paper reports:         {EXPECTED_OBS_PAPER:,}')
print(f'Excel row limit:       1,048,575')
if n_rows == 1_048_575:
    print('\n⚠ FILE IS TRUNCATED AT EXCEL ROW LIMIT')
    print(f'  Missing rows: {EXPECTED_OBS_PAPER - n_rows:,}')
print(f'\nUnique trader_ids: {df_raw["trader_id"].nunique():,} (paper: {EXPECTED_TRADERS_PAPER:,})')
print(f'Period range: {df_raw["period"].min()} — {df_raw["period"].max()}')

## 6. keep_country Investigation

In [ ]:
# Check 1: Fraction and distribution
kc = df_raw['keep_country']
print(f'keep_country value counts:')
print(kc.value_counts())
print(f'\nFraction kc=0: {(kc==0).mean():.2%}')

# Check 2: Is user_country_name null for kc=0?
kc0 = df_raw[kc == 0]
kc1 = df_raw[kc == 1]
print(f'\nkc=0 rows with null user_country_name: {kc0["user_country_name"].isna().mean():.1%}')
print(f'kc=1 rows with null user_country_name: {kc1["user_country_name"].isna().mean():.1%}')

# Show sample kc=0 rows
print('\nSample kc=0 rows:')
display(kc0[['trader_id', 'period', 'user_country_name', 'total_gain', 'total_self_trxn', 'keep_country']].head(10))

#  1,229,439 - paper
# 1,287,824 - 
# 1,196,100

In [ ]:
# Check 3: Compare outcomes
print('=== Comparison: kc=0 vs kc=1 ===')
for col in ['total_gain', 'total_self_trxn', 'total_trxn']:
    print(f'\n{col}:')
    print(f'  kc=0: mean={kc0[col].mean():.2f}, median={kc0[col].median():.2f}')
    print(f'  kc=1: mean={kc1[col].mean():.2f}, median={kc1[col].median():.2f}')

## 7. Column Diagnostics

In [ ]:
diag = verify_column_diagnostics(df_raw)
print('Column diagnostics:')
for k, v in diag.items():
    print(f'  {k}: {v}')

## 8. Clean Data and Verify

In [ ]:
df, stats = clean_data(df_raw, apply_keep_country=None)
print_cleaning_summary(stats)
checks = verify_counts(stats)
print(f'\nVerification: {checks}')

## 9. Panel Structure Verification

In [ ]:
rows_per_trader = df.groupby('trader_id').size()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(rows_per_trader, bins=46, color=COLORS['primary'], alpha=0.7, edgecolor='white')
ax.axvline(rows_per_trader.median(), color=COLORS['secondary'], linestyle='--', label=f'Median={rows_per_trader.median():.0f}')
ax.set_xlabel('Periods per Trader')
ax.set_ylabel('Number of Traders')
ax.set_title('Panel Balance: Periods per Trader')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Rows-per-trader: mean={rows_per_trader.mean():.1f}, median={rows_per_trader.median():.0f}')

The cleaned panel used by the analysis is written by `python run_pipeline.py --reload-data`, not by this notebook, so the pipeline's cache and cleaning statistics stay consistent.